# Pembanding TripoSR di Kaggle (RQ1)

Jalankan **TripoSR** pada subset foto yang sama dengan SF3D, lalu bandingkan di notebook evaluasi (RQ1 = perbandingan model, PRD §11).

**TripoSR vs SF3D:** lebih ringan, tetapi **hanya warna verteks** — tanpa PBR/delight/UV, jadi relight di viewer lemah. Ini justru bukti yang dicari RQ1: menunjukkan apa yang hilang tanpa material PBR.

**Sebelum mulai:** GPU T4 ON, Internet ON, attach dataset foto. (HF_TOKEN opsional — TripoSR publik.)

> Kernel TERPISAH dari SF3D: dep TripoSR (torchmcubes) bisa bentrok bila satu kernel.

In [ ]:
#@title 1. Cek GPU + temukan path dataset otomatis
import glob, os, sys
from collections import Counter
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
!nvidia-smi -L
imgs = []
for ext in ('jpg', 'jpeg', 'png', 'webp'):
    imgs += glob.glob(f'/kaggle/input/**/*.{ext}', recursive=True)
    imgs += glob.glob(f'/kaggle/input/**/*.{ext.upper()}', recursive=True)
counts = Counter(os.path.dirname(p) for p in imgs)
INPUT = counts.most_common(1)[0][0] if counts else ''
print('INPUT =', INPUT, '| jumlah di folder:', counts.get(INPUT, 0), '| total:', len(imgs))

In [ ]:
#@title 2. Pasang TripoSR + perbaiki ABI numpy/scipy. >>> RESTART setelah cell ini <<<
%cd /kaggle/working
import os
if not os.path.exists('tsr_repo'):
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git tsr_repo
!pip install -q -r tsr_repo/requirements.txt
# rembg butuh onnxruntime (tak selalu ikut requirements)
!pip install -q onnxruntime
# requirements TripoSR menurunkan numpy -> bentrok ABI dgn scipy bawaan Kaggle
# (ValueError: numpy.dtype size changed, Expected 96 got 88). Samakan pasangan:
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
# cupy Kaggle dibangun utk numpy 2.x -> rusak stlh numpy diturunkan. pymatting (dep
# rembg) pakai cupy utk akselerasi tapi except-nya tak tangkap ImportError -> bocor.
# Buang cupy: pymatting fallback ke CPU (rembg tetap jalan).
!pip uninstall -y -q cupy-cuda12x cupy-cuda11x cupy
print('Pasang OK.')
print('>>> SEKARANG RESTART KERNEL (menu Run > Restart), lalu jalankan cell 1,3,4,5 (LEWATI cell 2). <<<')

In [ ]:
#@title 3. Clone BERSIH repo tim (src terbaru) + path tsr + HF login (opsional)
REPO_URL = "https://github.com/eycoo/FP_GRAFKOM.git"  #@param {type:"string"}
import os, sys, shutil
sys.path.insert(0, '/kaggle/working/tsr_repo')  # re-add stlh restart kernel
import tsr; print('TripoSR import OK')
if os.path.exists('/kaggle/working/FP_GRAFKOM'):
    shutil.rmtree('/kaggle/working/FP_GRAFKOM')
!git clone -q $REPO_URL /kaggle/working/FP_GRAFKOM
!cd /kaggle/working/FP_GRAFKOM && git log -1 --oneline
sys.path.insert(0, '/kaggle/working/FP_GRAFKOM/pipeline/src')
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
    print('HF login OK')
except Exception as e:
    print('HF login dilewati (TripoSR publik):', str(e)[:60])

In [ ]:
#@title 4. Muat TripoSR + config + metadata kategori
import importlib, torch
import config, triposr, preprocess, run_pipeline, categorize
for _m in (config, triposr, preprocess, run_pipeline, categorize):
    importlib.reload(_m)
from config import PipelineCfg
from triposr import TripoSRReconstructor
from preprocess import load_rembg_session
from categorize import load_title_map, default_metadata_path

cfg = PipelineCfg.load('/kaggle/working/FP_GRAFKOM/pipeline/configs/default.yaml',
                       **{'export.out_dir': '/kaggle/working/outputs_triposr',
                          'model.name': 'stabilityai/TripoSR',
                          'model.config_name': 'config.yaml',
                          'model.weight_name': 'model.ckpt'})
MC_RES = 256   #@param {type:"integer"}
cfg.model.mc_resolution = MC_RES
cfg.model.chunk_size = 8192

reconstructor = TripoSRReconstructor(cfg.model)
title_map = load_title_map(default_metadata_path())
rembg_session = load_rembg_session()
print('device:', reconstructor.device, '| mc_resolution:', MC_RES, '| kategori:', len(title_map))
print('VRAM resident:', round(torch.cuda.memory_allocated()/1024**3, 2), 'GB')

In [ ]:
#@title 5. Inferensi batch TripoSR -> manifest_triposr.jsonl
from run_pipeline import collect_images, process_one
from export_glb import write_manifest
from pathlib import Path
import torch, gc

LIMIT = 5   #@param {type:"integer"}  # samakan dgn subset SF3D utk perbandingan adil
manifest = Path(cfg.export.out_dir) / 'manifest_triposr.jsonl'
images = collect_images(Path(INPUT))
if LIMIT:
    images = images[:LIMIT]
print(f'TripoSR proses {len(images)} foto (warna verteks, tanpa PBR)')
for i, img in enumerate(images, 1):
    gc.collect(); torch.cuda.empty_cache()
    try:
        rec = process_one(img, reconstructor, cfg, rembg_session, 'triposr', title_map)
        write_manifest(rec, manifest)
        print(f"[{i}/{len(images)}] {img.name[:36]} [{rec['category']}] -> {rec['seconds']}s, "
              f"VRAM {rec['peak_vram_gb']}GB, {rec['glb_bytes']//1024}KB")
    except Exception as e:
        torch.cuda.empty_cache()
        print(f'[{i}] GAGAL {img.name[:36]}: {str(e)[:80]}')
print('manifest:', manifest)

In [ ]:
#@title 6. Zip hasil TripoSR untuk diunduh
import shutil
shutil.make_archive('/kaggle/working/triposr_outputs', 'zip', '/kaggle/working/outputs_triposr')
print('Unduh: /kaggle/working/triposr_outputs.zip (tab Output)')
!ls -la /kaggle/working/outputs_triposr/glb | head

## Serah-terima
- `manifest_triposr.jsonl` + GLB → notebook evaluasi gabungkan dgn `manifest.json` SF3D (kolom `model` membedakan).
- Bandingkan: waktu, VRAM, ukuran GLB, dan **kualitas visual** (TripoSR tanpa PBR → cek apakah relight viewer memang lemah).